In [1]:
pip install transformers sentence-transformers langdetect pypdf2 gtts sounddevice scipy librosa openai-whisper


  Using cached librosa-0.11.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached audioread-3.0.1-py3-none-any.whl.metadata (8.4 kB)
  Using cached pooch-1.8.2-py3-none-any.whl.metadata (10 kB)
  Using cached lazy_loader-0.4-py3-none-any.whl.metadata (7.6 kB)
Using cached librosa-0.11.0-py3-none-any.whl (260 kB)
Using cached audioread-3.0.1-py3-none-any.whl (23 kB)
Using cached lazy_loader-0.4-py3-none-any.whl (12 kB)
Using cached pooch-1.8.2-py3-none-any.whl (64 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [librosa]

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
import PyPDF2
import numpy as np
from langdetect import detect
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
import sounddevice as sd
import scipy.io.wavfile as wav
import whisper
from gtts import gTTS
import os

In [3]:
!apt-get update -qq && apt-get install -y portaudio19-dev

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libportaudio2 libportaudiocpp0
Suggested packages:
  portaudio19-doc
The following NEW packages will be installed:
  libportaudio2 libportaudiocpp0 portaudio19-dev
0 upgraded, 3 newly installed, 0 to remove and 37 not upgraded.
Need to get 188 kB of archives.
After this operation, 927 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libportaudio2 amd64 19.6.0-1.1 [65.3 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libportaudiocpp0 amd64 19.6.0-1.1 [16.1 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 portaudio19-dev amd64 19.6.0-1.1 [106 kB]
Fetched 188 kB in 1s (32

In [ ]:
def load_pdf_text(pdf_path):
    text = ""
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + " "
    return text.strip()

# Load PDFs for three languages
pdf_english = "eng_doc.pdf"         
pdf_malayalam = "ml_doc.pdf"     
pdf_french = "fr_doc.pdf"           

text_english = load_pdf_text(pdf_english)
text_malayalam = load_pdf_text(pdf_malayalam)
text_french = load_pdf_text(pdf_french)

In [21]:
def split_into_chunks(text, chunk_size=150):
    words = text.split()
    chunks = [' '.join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]
    return chunks

chunks_en = split_into_chunks(text_english)
chunks_ml = split_into_chunks(text_malayalam)
chunks_fr = split_into_chunks(text_french)

In [22]:
embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

emb_en = embedder.encode(chunks_en)
emb_ml = embedder.encode(chunks_ml)
emb_fr = embedder.encode(chunks_fr)

In [23]:
def retrieve_top_chunks(question, chunks, embeddings, top_k=3):
    q_emb = embedder.encode([question])
    sims = cosine_similarity(q_emb, embeddings)[0]
    top_idx = np.argsort(sims)[-top_k:][::-1]
    return [chunks[i] for i in top_idx]


In [24]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Foundation English (FLAN-T5)
tokenizer_en = AutoTokenizer.from_pretrained("google/flan-t5-base")
model_en = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base").to(device)

# Indic Model (Malayalam) - IndicBERT
tokenizer_ml = AutoTokenizer.from_pretrained("ai4bharat/IndicBERTv2-MLM-only")
model_ml = AutoModel.from_pretrained("ai4bharat/IndicBERTv2-MLM-only").to(device)

# International (French) - CamemBERT
tokenizer_fr = AutoTokenizer.from_pretrained("camembert-base")
model_fr = AutoModel.from_pretrained("camembert-base").to(device)

In [ ]:
# Generative QA for English (FLAN-T5)
def generate_answer_t5(question, context, tokenizer, model):
    prompt = f"question: {question} context: {context}"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    outputs = model.generate(**inputs, max_length=200)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Simple Extractive QA for Malayalam & French (Baseline)
def generate_answer_extractive(question, top_chunks):
    
    return top_chunks[0]


In [26]:
def record_audio(filename="question.wav", duration=5):
    fs = 16000
    print("Recording...")
    audio = sd.rec(int(duration * fs), samplerate=fs, channels=1)
    sd.wait()
    wav.write(filename, fs, (audio * 32767).astype(np.int16))
    print("Recording complete!")

def speech_to_text(filename="question.wav"):
    model = whisper.load_model("base")
    result = model.transcribe(filename)
    return result['text']

In [28]:
def speak(text, lang_code="en"):
    tts = gTTS(text=text, lang=lang_code)
    tts.save("answer.mp3")
    os.system("start answer.mp3" if os.name == "nt" else "mpg123 answer.mp3")

In [ ]:
def qa_system(language="english"):
    print("\n--- Speak your question ---")
    record_audio(duration=5)
    question = speech_to_text()
    print("Recognized Question:", question)

    if language == "english":
        chunks, emb = chunks_en, emb_en
        top_chunks = retrieve_top_chunks(question, chunks, emb)
        answer = generate_answer_t5(question, " ".join(top_chunks), tokenizer_en, model_en)
        speak(answer, "en")

    elif language == "malayalam":
        chunks, emb = chunks_ml, emb_ml
        top_chunks = retrieve_top_chunks(question, chunks, emb)
        answer = generate_answer_extractive(question, top_chunks)
        speak(answer, "ml")  # Malayalam

    else:  # International (French)
        chunks, emb = chunks_fr, emb_fr
        top_chunks = retrieve_top_chunks(question, chunks, emb)
        answer = generate_answer_extractive(question, top_chunks)
        speak(answer, "fr")  

    print("Answer:", answer)
    return answer


In [32]:
# Run English QA
qa_system(language="english")

# Run Malayalam QA
qa_system(language="malayalam")

# Run French QA
qa_system(language="international")


--- Speak your question ---
Recording...
Recording complete!


/Users/abhijith/Christ/LLM/venv310/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Recognized Question:  What is artificial intelligence?


High Performance MPEG 1.0/2.0/2.5 Audio Player for Layers 1, 2 and 3
	version 1.33.0; written and copyright by Michael Hipp and others
	free software (LGPL) without any warranty but with best wishes

Playing MPEG stream 1 of 1: answer.mp3 ...

MPEG 2.0 L III cbr64 24000 mono

[0:06] Decoding of answer.mp3 finished.


Answer: ability of a computer or machine to perform tasks that typically require human intelligence

--- Speak your question ---
Recording...
Recording complete!


...

/Users/abhijith/Christ/LLM/venv310/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Recognized Question:  Qu'est-ce que l'intelligence artificielle ?


High Performance MPEG 1.0/2.0/2.5 Audio Player for Layers 1, 2 and 3
	version 1.33.0; written and copyright by Michael Hipp and others
	free software (LGPL) without any warranty but with best wishes

Playing MPEG stream 1 of 1: answer.mp3 ...

MPEG 2.0 L III cbr64 24000 mono

[0:22] Decoding of answer.mp3 finished.


Answer: L'intelligence artificielle désigne la capacité d'un ordinateur ou d'une machine à effectuer des tâches qui requièrent généralement l'intelligence humaine. Cela inclut des capacités telles que l'apprentissage, le raisonnement, la résolution de problèmes et la prise de décision. L'IA vise essentiellement à créer des systèmes capables d'imiter l'humain.


"L'intelligence artificielle désigne la capacité d'un ordinateur ou d'une machine à effectuer des tâches qui requièrent généralement l'intelligence humaine. Cela inclut des capacités telles que l'apprentissage, le raisonnement, la résolution de problèmes et la prise de décision. L'IA vise essentiellement à créer des systèmes capables d'imiter l'humain."